# 08 Proyecto: entorno de desarrollo completo con docker-compose


Python + PostgreSQL + pgAdmin todo en docker-compose. Un mini pipeline ETL que corre 100% en Docker.



## [01] Añade un healthcheck a PostgreSQL

En vez de usar la función esperar_postgres() en Python, configura un healthcheck en docker-compose para que Docker sepa cuándo PostgreSQL está realmente listo. El ETL debe depender de que la DB esté "healthy".

- Estás en la carpeta pipeline/. Escribe el docker-compose.yml con el healthcheck de postgres y el etl que depende de él. Valida con docker compose config.

- **[NOTA]** Aquí se valida que el healthcheck y la dependencia estén DECLARADOS correctamente en el YAML; el simulador no ejecuta el pg_isready real.



```yml
services:
  postgres:
    image: postgres:16
    environment:
      POSTGRES_USER: etl_user
      POSTGRES_PASSWORD: etl_secreto
      POSTGRES_DB: ventas
    ports:
      - "5432:5432"
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U etl_user -d ventas"]
      interval: 5s
      timeout: 5s
      retries: 5
  etl:
    build: .
    depends_on:
      postgres:
        condition: service_healthy
```

**Estrategia:** usar healthcheck nativo de Docker para que depends_on espere a que PostgreSQL esté REALMENTE listo (no solo que el contenedor haya arrancado).

1. `pg_isready` → Utilidad incluida en la imagen de PostgreSQL que verifica si acepta conexiones.
2. `interval: 5s` → Chequea cada 5 segundos.
3. `timeout: 5s` → Si no responde en 5s, cuenta como fallo.
4. `retries: 5` → Después de 5 fallos seguidos, marca el servicio como "unhealthy".
5. `condition: service_healthy` → El ETL solo arranca cuando PostgreSQL pasa el healthcheck.

**Error común:** 
- depends_on SIN condition solo espera que el contenedor arranque. 
- PostgreSQL tarda 2-5s en inicializarse después de arrancar.

## [02] Extender el pipeline con un reporte

Modifica el pipeline para que, después de cargar los datos, genere un reporte JSON con estadísticas: 

1. Total de ventas
2. Número de filas
3. La ciudad con más ventas. 

Guárdalo en `/app/output/reporte.json`


In [ ]:
import json, psycopg2, os

def generar_reporte():
    """Genera un reporte JSON con estadística del pipeline."""
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST", "postgres"),
        post=5432,
        user=os.getenv("DB_USER", "etl_user"),
        password=os.getenv("DB_PASS", "etl_secreto"),
        database=os.getenv("DB_NAME", "ventas")
    )

    cursor = conn.cursor()
    
    # Total de filas cargadas
    cursor.execute("""
        SELECT COUNT(*) FROM ventas_limpias
    """)

    total_filas = cursor.fetchall()[0]

    # Suma total de ventas
    cursor.execute("""
        SELECT SUM(total) FROM ventas_limpias
    """)

    total_ventas = float(cursor.fetchall()[0] or 0)

    # Ciudad con más ventas
    cursor.execute("""
        SELECT ciudad, SUM(total) as t FROM ventas_limpias
        GROUP BY ciudad
        ORDER BY t 
        DESC LIMIT 1
    """)

    top_ciudad = cursor.fetchone()

    reporte = {
        "total_filas": total_filas,
        "total_ventas": round(total_ventas, 2),
        "top_ciudad": {"nombre": top_ciudad[0],
                       "total": float(top_ciudad[1])}
    }

    conn.close()

    # Guardar reporte

    ruta_reporte = "/app/output/"
    os.makedirs(ruta_reporte, exist_ok=True)

    with open(f"{ruta_reporte}reporte.json", "w") as f:
        json.dump(reporte, f, indent=2, ensure_ascii=False)

    print(f"Reporte generado: {reporte}")

# generar_reporte()


Estrategia: ejecutar queries de resumen sobre los datos ya cargados y exportar resultados como JSON.

1. `COUNT(*)` → Total de filas que pasaron la validación del ETL.
2. `SUM(total)` → Revenue total. El `or 0` previene error si la tabla está vacía (SUM retorna NULL).
3. `GROUP BY ciudad ORDER BY t DESC LIMIT 1` → La ciudad con mayor facturación. Es un patrón TOP-N.
4. `os.makedirs(..., exist_ok=True)` → Crea /app/output si no existe, sin error si ya existe.
5. `json.dump(..., ensure_ascii=False)` → Permite caracteres UTF-8 (acentos, eñes) en el JSON.

Error común: olvidar `float()` al extraer Decimals de PostgreSQL. psycopg2 retorna objetos Decimal que json.dump no serializa.

## [03] Dockerfile multi-stage para producción

El valor de un multi-stage es el tamaño de la imagen final, y aquí no podemos ejecutarlo para medir los MB sin inventárnoslos. 

Así que lo razonas: 
1. Escribe la versión de producción del Dockerfile —builder para instalar
2. Runtime para ejecutar 
3. Un usuario sin privilegios que pueda escribir en /app/output
4. Explica cada decisión. 
5. Compara luego con la solución.

Partes de este Dockerfile de desarrollo (funciona, pero es pesado e inseguro):


```Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY etl/ ./etl/
COPY data/ ./data/
CMD ["python", "etl/pipeline.py"]
```

```Dockerfile
# Stage 1: builder
FROM python:3.12-slim AS builder

WORKDIR /builder

COPY requirements.txt .

RUN pip install --no-cache-dir --target=/build/deps -r requirements.txt

# Stage 2: runtime
FROM python:3.12-slim AS runtime

WORKDIR /app

COPY --from=builder /app/deps/usr/local/lib/python3.12/site-packages/

COPY etl/ ./etl/
COPY data/ ./data/

RUN useradd -m appuser && mkdir -p /app/output && chown -R appuser /app

CMD ["python", "etl/pipeline.py"]
```

## [04] El .dockerignore perfecto

Tu proyecto tiene archivos que NO deberían estar en la imagen Docker: 

- Entornos virtuales
- Archivos de test
- Documentación
- Secrets
  
Escribe el .dockerignore completo.


```.dockerignore
# .dockerignore - archivos que NO van en la imagen Docker

# Entornos virtuales de Python
.venv/
venv/
__pycache__/
*.pyc

# Control de versiones
.git/
.gitignore

# Archivos de IDE
.vscode/
.idea/

# Archivos de testing
tests/
pytest.ini

# Documentación
*.md
docs/

# Variables de entorno con secretos
.env
.env.local

# Archivos del sistema operativo
.DS_Store
Thumbs.db

# Output/resultados locales
output/
*.log

```

1. .venv/ y __pycache__/ → Se regeneran dentro de la imagen. Copiarlos es redundante.
2. .git/ → Puede pesar cientos de MB. NUNCA lo necesitas en la imagen.
3. .vscode/ → Configuración de tu IDE personal. Irrelevante en producción.
4. tests/ → Los tests se ejecutan en CI, no dentro de la imagen de producción.
5. .env → SECRETOS. Si los copias a la imagen, cualquiera que descargue la imagen los ve.
6. .DS_Store / Thumbs.db → Archivos ocultos del OS que contaminarían la imagen.
7. output/ → Resultados locales que no deben ir en la imagen.